# G1 Humanoid Robot - Rolling Horizon MPC Balancing

This notebook implements Model Predictive Control (MPC) with rolling horizon for balancing the G1 humanoid robot.

## Key Differences from Previous Approach:
- **Rolling Horizon**: Instead of one large factor graph for the entire trajectory, we build small factor graphs with short time horizons (0.5s)
- **Receding Horizon**: After each optimization, we apply the first control step and rebuild the factor graph from the current state
- **Online Re-planning**: Uses current MuJoCo state as the starting point for each new optimization

## Workflow:
1. Load G1 robot model
2. Initialize MuJoCo simulation
3. **MPC Loop** (repeat until simulation complete):
   - Build factor graph for short horizon (0.5s)
   - Optimize trajectory from current state
   - Apply first control action
   - Step MuJoCo simulation
   - Use new state for next iteration
4. Visualize results with Plotly

## Parameters:
- Total simulation time: 5.0 seconds
- MPC horizon: 0.5 seconds
- Control frequency: 50 Hz (0.02s per step)

## 1. Imports and Setup

In [1]:
import numpy as np
import gtsam
import gtdynamics as gtd
import mujoco
import mujoco.viewer
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mediapy as media
from typing import List, Tuple, Dict
import time as pytime

print("✓ Imports successful!")
print(f"GTDynamics version: {gtd.__version__ if hasattr(gtd, '__version__') else 'Unknown'}")
print(f"GTSAM version: {gtsam.__version__ if hasattr(gtsam, '__version__') else 'Unknown'}")
print(f"MuJoCo version: {mujoco.__version__ if hasattr(mujoco, '__version__') else 'Unknown'}")

✓ Imports successful!
GTDynamics version: Unknown
GTSAM version: Unknown
MuJoCo version: 3.3.2


## 2. Configuration Parameters

In [2]:
# File paths
URDF_PATH = '../../models/urdfs/g1_description/g1_23dof.urdf'
MJCF_PATH = '../../models/urdfs/g1_description/g1_23dof.xml'

# MPC Parameters
TOTAL_SIM_TIME = 5.0      # Total simulation duration (seconds)
MPC_HORIZON = 0.5         # MPC optimization horizon (seconds)
CONTROL_DT = 0.02         # Control timestep (50 Hz)
MPC_STEPS = int(MPC_HORIZON / CONTROL_DT)  # Steps per MPC optimization

# Physics parameters
GRAVITY = np.array([0, 0, -9.81])
FRICTION_COEF = 1.0

# Cost model parameters (noise models)
SIGMA_DYNAMICS = 1e-5
SIGMA_OBJECTIVES = 1e-3
SIGMA_TORQUE = 1e-4

# Rendering parameters
VIDEO_WIDTH = 800
VIDEO_HEIGHT = 600
RENDER_FPS = 30

print(f"MPC Configuration:")
print(f"  Total simulation time: {TOTAL_SIM_TIME}s")
print(f"  MPC horizon: {MPC_HORIZON}s ({MPC_STEPS} steps)")
print(f"  Control timestep: {CONTROL_DT}s ({1/CONTROL_DT:.0f} Hz)")
print(f"  Total MPC iterations: {int(TOTAL_SIM_TIME / CONTROL_DT)}")

MPC Configuration:
  Total simulation time: 5.0s
  MPC horizon: 0.5s (25 steps)
  Control timestep: 0.02s (50 Hz)
  Total MPC iterations: 250


## 3. Load Robot Model

In [3]:
# Load G1 robot from URDF
robot = gtd.CreateRobotFromFile(URDF_PATH)

num_links = robot.numLinks()
num_joints = robot.numJoints()

print(f"✓ Robot loaded: {num_links} links, {num_joints} joints")

# Find important link IDs
pelvis_link_id = None
left_ankle_id = None
right_ankle_id = None

link_names = []
for i, link in enumerate(robot.links()):
    link_names.append(link.name())
    if link.name() == "pelvis":
        pelvis_link_id = i
    elif link.name() == "left_ankle_roll_link":
        left_ankle_id = i
    elif link.name() == "right_ankle_roll_link":
        right_ankle_id = i

joint_names = [joint.name() for joint in robot.joints()]

print(f"  Pelvis link ID: {pelvis_link_id}")
print(f"  Left ankle ID: {left_ankle_id}")
print(f"  Right ankle ID: {right_ankle_id}")

✓ Robot loaded: 24 links, 23 joints
  Pelvis link ID: 11
  Left ankle ID: 1
  Right ankle ID: 13


## 4. Define Contact Points

In [4]:
# Contact points on feet (center of foot)
left_foot_link = robot.link("left_ankle_roll_link")
right_foot_link = robot.link("right_ankle_roll_link")

# Center of foot contact point
foot_center = np.array([0.035, 0.0, -0.03])

contact_points = [
    gtd.PointOnLink(left_foot_link, foot_center),
    gtd.PointOnLink(right_foot_link, foot_center)
]

print(f"✓ Created {len(contact_points)} contact points at foot centers")
print(f"  Friction coefficient: {FRICTION_COEF}")

✓ Created 2 contact points at foot centers
  Friction coefficient: 1.0


## 5. MuJoCo Setup and Joint Mapping

In [5]:
# Load MuJoCo model
mj_model = mujoco.MjModel.from_xml_path(MJCF_PATH)
mj_data = mujoco.MjData(mj_model)

print(f"✓ MuJoCo model loaded")
print(f"  nq={mj_model.nq}, nv={mj_model.nv}, nu={mj_model.nu}")

# Create mapping from GTDynamics to MuJoCo joint indices
gtd_to_mj_pos = {}
gtd_to_mj_vel = {}

mj_joint_names = []
for i in range(mj_model.njnt):
    joint_name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_JOINT, i)
    if joint_name:
        mj_joint_names.append(joint_name)

for gtd_idx, joint in enumerate(robot.joints()):
    gtd_joint_name = joint.name()
    for mj_idx, mj_name in enumerate(mj_joint_names):
        if gtd_joint_name == mj_name:
            joint_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_JOINT, mj_name)
            gtd_to_mj_pos[gtd_idx] = mj_model.jnt_qposadr[joint_id]
            gtd_to_mj_vel[gtd_idx] = mj_model.jnt_dofadr[joint_id]
            break

print(f"✓ Mapped {len(gtd_to_mj_pos)} joints GTD↔MuJoCo")

# Set MuJoCo timestep
mj_model.opt.timestep = CONTROL_DT

✓ MuJoCo model loaded
  nq=30, nv=29, nu=23
✓ Mapped 23 joints GTD↔MuJoCo


## 6. Helper Functions

In [6]:
def get_current_state_from_mujoco(mj_data, gtd_to_mj_pos, gtd_to_mj_vel, num_joints):
    """Extract current joint angles and velocities from MuJoCo."""
    q = np.zeros(num_joints)
    v = np.zeros(num_joints)
    
    for gtd_idx in range(num_joints):
        if gtd_idx in gtd_to_mj_pos:
            q[gtd_idx] = mj_data.qpos[gtd_to_mj_pos[gtd_idx]]
            v[gtd_idx] = mj_data.qvel[gtd_to_mj_vel[gtd_idx]]
    
    # Get base pose (first 7 elements: position + quaternion)
    base_pos = mj_data.qpos[:3].copy()
    base_quat_wxyz = mj_data.qpos[3:7].copy()  # [w, x, y, z]
    base_rot = gtsam.Rot3.Quaternion(base_quat_wxyz[0], base_quat_wxyz[1], 
                                      base_quat_wxyz[2], base_quat_wxyz[3])
    base_pose = gtsam.Pose3(base_rot, base_pos)
    
    # Get base twist (first 6 elements of qvel)
    base_twist = mj_data.qvel[:6].copy()
    
    return q, v, base_pose, base_twist


def apply_control_to_mujoco(mj_data, torques, gtd_to_mj_pos, num_joints):
    """Apply torque controls to MuJoCo actuators."""
    for j in range(min(num_joints, len(torques))):
        if j < mj_data.ctrl.shape[0]:
            mj_data.ctrl[j] = torques[j]


print("✓ Helper functions defined:")

✓ Helper functions defined:


## 7. MPC Optimization Function

In [9]:
def build_and_optimize_mpc(robot, current_q, current_v, current_base_pose, current_base_twist,
                          contact_points, mpc_steps, dt, target_base_height=0.8):
    """
    Build factor graph for MPC horizon and optimize.
    
    Returns:
        result: Optimized values
        graph: Factor graph (for debugging)
        success: Whether optimization succeeded
    """
    # Noise models
    dynamics_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, SIGMA_DYNAMICS)
    dynamics_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, SIGMA_DYNAMICS)
    objectives_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, SIGMA_OBJECTIVES)
    torque_model = gtsam.noiseModel.Isotropic.Sigma(1, SIGMA_TORQUE)
    
    # Create graph builder
    opt_params = gtd.OptimizerSetting(SIGMA_DYNAMICS)
    graph_builder = gtd.DynamicsGraph(opt_params, GRAVITY, None)
    
    # Build trajectory factor graph
    collocation_scheme = gtd.CollocationScheme.Trapezoidal
    graph = graph_builder.trajectoryFG(robot, mpc_steps, dt, 
                                        collocation_scheme, contact_points, FRICTION_COEF)
    
    # Add boundary conditions - INITIAL STATE (current state from MuJoCo)
    for j in range(num_joints):
        graph.addPriorDouble(gtd.JointAngleKey(j, 0), current_q[j], dynamics_model_1)
        graph.addPriorDouble(gtd.JointVelKey(j, 0), current_v[j], dynamics_model_1)
    
    # Initial base pose and twist
    graph.add(gtsam.PriorFactorPose3(gtd.PoseKey(pelvis_link_id, 0), 
                                     current_base_pose, dynamics_model_6))
    graph.add(gtsam.PriorFactorVector(gtd.TwistKey(pelvis_link_id, 0), 
                                      current_base_twist, dynamics_model_6))
    
    # Constrain base pose throughout horizon (maintain standing)
    target_base_pose = gtsam.Pose3(gtsam.Rot3(), np.array([0.0, 0.0, target_base_height]))
    zero_twist = np.zeros(6)
    
    relaxed_pose_model = gtsam.noiseModel.Isotropic.Sigma(6, 1e-5)
    relaxed_twist_model = gtsam.noiseModel.Isotropic.Sigma(6, 1e-5)
    
    for t in range(1, mpc_steps + 1):
        graph.add(gtsam.PriorFactorPose3(gtd.PoseKey(pelvis_link_id, t), 
                                         target_base_pose, relaxed_pose_model))
        graph.add(gtsam.PriorFactorVector(gtd.TwistKey(pelvis_link_id, t), 
                                          zero_twist, relaxed_twist_model))
    
    # Relaxed joint objectives (prefer standing pose but allow adjustment)
    target_joint_angles = np.zeros(num_joints)
    relaxed_joint_model = gtsam.noiseModel.Isotropic.Sigma(1, 1e-2)
    
    for t in range(1, mpc_steps + 1):
        for j in range(num_joints):
            graph.addPriorDouble(gtd.JointAngleKey(j, t), target_joint_angles[j], relaxed_joint_model)
            graph.addPriorDouble(gtd.JointVelKey(j, t), 0.0, relaxed_joint_model)
    
    # Minimize torques
    for t in range(mpc_steps + 1):
        for j in range(num_joints):
            graph.add(gtd.MinTorqueFactor(gtd.TorqueKey(j, t), torque_model))
    
    # Initialize values
    initializer = gtd.Initializer()
    init_values = initializer.ZeroValuesTrajectory(robot, mpc_steps, 0, 0.0, contact_points)
    
    # Set initial state from current state
    for j in range(num_joints):
        init_values.update(gtd.JointAngleKey(j, 0), current_q[j])
        init_values.update(gtd.JointVelKey(j, 0), current_v[j])
    
    # Set base pose/twist
    for t in range(mpc_steps + 1):
        pose_key = gtd.PoseKey(pelvis_link_id, t)
        twist_key = gtd.TwistKey(pelvis_link_id, t)
        
        if init_values.exists(pose_key):
            init_values.erase(pose_key)
        init_values.insert(pose_key, target_base_pose if t > 0 else current_base_pose)
        
        if init_values.exists(twist_key):
            init_values.erase(twist_key)
        init_values.insert(twist_key, zero_twist if t > 0 else current_base_twist)
    
    # Optimize
    params = gtsam.LevenbergMarquardtParams()
    params.setVerbosity("SILENT")
    params.setAbsoluteErrorTol(1e-6)
    params.setRelativeErrorTol(1e-6)
    params.setMaxIterations(50)
    
    optimizer = gtsam.LevenbergMarquardtOptimizer(graph, init_values, params)
    result = optimizer.optimize()
    
    final_error = graph.error(result)
    success = final_error < 1e-3
    
    return result, graph, success, final_error


print("✓ MPC optimization function defined")

✓ MPC optimization function defined


## 8. Main MPC Loop

In [10]:
# Reset MuJoCo to initial standing configuration
mujoco.mj_resetData(mj_model, mj_data)

# Set initial joint angles (neutral standing pose)
initial_joint_angles = np.zeros(num_joints)
for gtd_idx in range(num_joints):
    if gtd_idx in gtd_to_mj_pos:
        mj_data.qpos[gtd_to_mj_pos[gtd_idx]] = initial_joint_angles[gtd_idx]

# Initialize data storage
data_log = {
    'time': [],
    'base_position': [],
    'base_height': [],
    'joint_angles': [[] for _ in range(num_joints)],
    'joint_velocities': [[] for _ in range(num_joints)],
    'applied_torques': [[] for _ in range(num_joints)],
    'optimization_errors': [],
    'optimization_times': [],
    'success_flags': []
}

# Rendering setup
frames = []
renderer = mujoco.Renderer(mj_model, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)

# MPC loop
total_iterations = int(TOTAL_SIM_TIME / CONTROL_DT)
print(f"\nStarting MPC loop: {total_iterations} iterations")
print(f"Building new factor graph every {MPC_STEPS} steps ({MPC_HORIZON}s)")
print("=" * 60)

for iteration in range(total_iterations):
    current_time = iteration * CONTROL_DT
    
    # Get current state from MuJoCo
    mujoco.mj_forward(mj_model, mj_data)
    current_q, current_v, current_base_pose, current_base_twist = get_current_state_from_mujoco(
        mj_data, gtd_to_mj_pos, gtd_to_mj_vel, num_joints
    )
    
    # Build and optimize MPC problem
    opt_start = pytime.time()
    result, graph, success, error = build_and_optimize_mpc(
        robot, current_q, current_v, current_base_pose, current_base_twist,
        contact_points, MPC_STEPS, CONTROL_DT
    )
    opt_time = pytime.time() - opt_start
    
    # Extract first control action (torques at t=0)
    torques = np.array([gtd.Torque(result, j, 0) for j in range(num_joints)])
    
    # Apply control to MuJoCo
    apply_control_to_mujoco(mj_data, torques, gtd_to_mj_pos, num_joints)
    
    # Step simulation
    mujoco.mj_step(mj_model, mj_data)
    
    # Log data
    base_pos = mj_data.qpos[:3].copy()
    data_log['time'].append(current_time)
    data_log['base_position'].append(base_pos)
    data_log['base_height'].append(base_pos[2])
    data_log['optimization_errors'].append(error)
    data_log['optimization_times'].append(opt_time)
    data_log['success_flags'].append(success)
    
    for j in range(num_joints):
        data_log['joint_angles'][j].append(current_q[j])
        data_log['joint_velocities'][j].append(current_v[j])
        data_log['applied_torques'][j].append(torques[j])
    
    # Render frames
    if iteration % max(1, int(total_iterations / (TOTAL_SIM_TIME * RENDER_FPS))) == 0:
        renderer.update_scene(mj_data)
        pixels = renderer.render()
        frames.append(pixels)
    
    # Progress updates
    if (iteration + 1) % 25 == 0:
        avg_opt_time = np.mean(data_log['optimization_times'][-25:])
        success_rate = np.mean(data_log['success_flags'][-25:]) * 100
        print(f"[{iteration+1:3d}/{total_iterations}] t={current_time:.2f}s | "
              f"Opt: {avg_opt_time*1000:.1f}ms | Success: {success_rate:.0f}% | "
              f"Height: {base_pos[2]:.3f}m")

renderer.close()
print("=" * 60)
print(f"✓ MPC simulation complete!")
print(f"  Total iterations: {total_iterations}")
print(f"  Average optimization time: {np.mean(data_log['optimization_times'])*1000:.2f}ms")
print(f"  Success rate: {np.mean(data_log['success_flags'])*100:.1f}%")


Starting MPC loop: 250 iterations
Building new factor graph every 25 steps (0.5s)
[ 25/250] t=0.48s | Opt: 983.4ms | Success: 0% | Height: 0.766m
[ 25/250] t=0.48s | Opt: 983.4ms | Success: 0% | Height: 0.766m
[ 50/250] t=0.98s | Opt: 1066.9ms | Success: 0% | Height: 0.353m
[ 50/250] t=0.98s | Opt: 1066.9ms | Success: 0% | Height: 0.353m
[ 75/250] t=1.48s | Opt: 956.5ms | Success: 0% | Height: 0.126m
[ 75/250] t=1.48s | Opt: 956.5ms | Success: 0% | Height: 0.126m


KeyboardInterrupt: 

## 9. Visualize Simulation Video

In [ ]:
media.show_video(frames, fps=RENDER_FPS, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)

## 10. Plot Base Position and Height

In [ ]:
time = data_log['time']
base_x = [pos[0] for pos in data_log['base_position']]
base_y = [pos[1] for pos in data_log['base_position']]
base_z = data_log['base_height']

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=['Base X Position', 'Base Y Position', 'Base Z Position (Height)'],
    vertical_spacing=0.08,
    shared_xaxes=True
)

# X position
fig.add_trace(
    go.Scatter(x=time, y=base_x, mode='lines', name='X',
               line=dict(width=2, color='red')),
    row=1, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=1, col=1)

# Y position
fig.add_trace(
    go.Scatter(x=time, y=base_y, mode='lines', name='Y',
               line=dict(width=2, color='green')),
    row=2, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=2, col=1)

# Z position (height)
fig.add_trace(
    go.Scatter(x=time, y=base_z, mode='lines', name='Z',
               line=dict(width=2, color='blue')),
    row=3, col=1
)
fig.add_hline(y=0.8, line_dash="dash", line_color="orange", opacity=0.5, row=3, col=1,
              annotation_text="Target (0.8m)", annotation_position="right")

fig.update_yaxes(title_text="X (m)", row=1, col=1)
fig.update_yaxes(title_text="Y (m)", row=2, col=1)
fig.update_yaxes(title_text="Z (m)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)

fig.update_layout(
    height=700,
    title_text="Base Link Position Over Time (MPC)",
    showlegend=False
)

fig.show()

print(f"Base position stats:")
print(f"  X: [{min(base_x):.4f}, {max(base_x):.4f}] m, final={base_x[-1]:.4f}m")
print(f"  Y: [{min(base_y):.4f}, {max(base_y):.4f}] m, final={base_y[-1]:.4f}m")
print(f"  Z: [{min(base_z):.4f}, {max(base_z):.4f}] m, final={base_z[-1]:.4f}m")

## 11. Plot MPC Performance Metrics

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Optimization Error', 'Optimization Time'],
    vertical_spacing=0.12,
    shared_xaxes=True
)

# Optimization error
fig.add_trace(
    go.Scatter(x=time, y=data_log['optimization_errors'], mode='lines',
               name='Error', line=dict(width=2, color='red')),
    row=1, col=1
)

# Optimization time
opt_times_ms = [t * 1000 for t in data_log['optimization_times']]
fig.add_trace(
    go.Scatter(x=time, y=opt_times_ms, mode='lines',
               name='Time', line=dict(width=2, color='blue')),
    row=2, col=1
)
fig.add_hline(y=CONTROL_DT*1000, line_dash="dash", line_color="red", opacity=0.5, row=2, col=1,
              annotation_text=f"Real-time limit ({CONTROL_DT*1000:.0f}ms)", 
              annotation_position="right")

fig.update_yaxes(title_text="Error", type="log", row=1, col=1)
fig.update_yaxes(title_text="Time (ms)", row=2, col=1)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)

fig.update_layout(
    height=600,
    title_text="MPC Optimization Performance",
    showlegend=False
)

fig.show()

print(f"Optimization performance:")
print(f"  Avg error: {np.mean(data_log['optimization_errors']):.2e}")
print(f"  Avg time: {np.mean(data_log['optimization_times'])*1000:.2f}ms")
print(f"  Max time: {np.max(data_log['optimization_times'])*1000:.2f}ms")
print(f"  Real-time capable: {np.max(data_log['optimization_times']) < CONTROL_DT}")

## 12. Plot Joint Angles

In [ ]:
# Plot a subset of important joints
important_joints = [
    ('left_hip_pitch_joint', 'Left Hip Pitch'),
    ('right_hip_pitch_joint', 'Right Hip Pitch'),
    ('left_knee_joint', 'Left Knee'),
    ('right_knee_joint', 'Right Knee'),
    ('left_ankle_pitch_joint', 'Left Ankle Pitch'),
    ('right_ankle_pitch_joint', 'Right Ankle Pitch'),
]

# Find joint indices
important_indices = []
important_labels = []
for joint_name, label in important_joints:
    try:
        idx = joint_names.index(joint_name)
        important_indices.append(idx)
        important_labels.append(label)
    except ValueError:
        pass

n_plots = len(important_indices)
fig = make_subplots(
    rows=n_plots, cols=1,
    subplot_titles=important_labels,
    vertical_spacing=0.02,
    shared_xaxes=True
)

for i, (idx, label) in enumerate(zip(important_indices, important_labels)):
    angles_deg = np.rad2deg(data_log['joint_angles'][idx])
    fig.add_trace(
        go.Scatter(x=time, y=angles_deg, mode='lines',
                   line=dict(width=2), showlegend=False),
        row=i+1, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=i+1, col=1)
    fig.update_yaxes(title_text="deg", row=i+1, col=1, title_font=dict(size=10))

fig.update_xaxes(title_text="Time (s)", row=n_plots, col=1)
fig.update_layout(
    height=150*n_plots,
    title_text="Important Joint Angles",
    showlegend=False
)

fig.show()

## 13. Plot Applied Torques

In [ ]:
# Plot torques for the same important joints
fig = make_subplots(
    rows=n_plots, cols=1,
    subplot_titles=important_labels,
    vertical_spacing=0.02,
    shared_xaxes=True
)

for i, (idx, label) in enumerate(zip(important_indices, important_labels)):
    torques = data_log['applied_torques'][idx]
    fig.add_trace(
        go.Scatter(x=time, y=torques, mode='lines',
                   line=dict(width=2), showlegend=False),
        row=i+1, col=1
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=i+1, col=1)
    fig.update_yaxes(title_text="N·m", row=i+1, col=1, title_font=dict(size=10))

fig.update_xaxes(title_text="Time (s)", row=n_plots, col=1)
fig.update_layout(
    height=150*n_plots,
    title_text="Applied Joint Torques (MPC)",
    showlegend=False
)

fig.show()

# Print torque statistics
print("\nTorque statistics (important joints):")
for idx, label in zip(important_indices, important_labels):
    torques = np.array(data_log['applied_torques'][idx])
    print(f"  {label:20s}: Max={np.max(np.abs(torques)):6.2f} N·m, "
          f"Avg={np.mean(np.abs(torques)):6.2f} N·m")

## 14. Summary Statistics

In [ ]:
print("=" * 70)
print("MPC BALANCING SIMULATION SUMMARY")
print("=" * 70)

print(f"\n📊 Simulation Parameters:")
print(f"  Total time: {TOTAL_SIM_TIME}s")
print(f"  MPC horizon: {MPC_HORIZON}s ({MPC_STEPS} steps)")
print(f"  Control frequency: {1/CONTROL_DT:.0f} Hz")
print(f"  Total MPC iterations: {len(data_log['time'])}")

print(f"\n🎯 Balancing Performance:")
print(f"  Initial height: {data_log['base_height'][0]:.4f}m")
print(f"  Final height: {data_log['base_height'][-1]:.4f}m")
print(f"  Height drift: {data_log['base_height'][-1] - data_log['base_height'][0]:.4f}m")
print(f"  Max height deviation: {max(abs(h - 0.8) for h in data_log['base_height']):.4f}m")
print(f"  Lateral drift (X): {abs(base_x[-1]):.4f}m")
print(f"  Lateral drift (Y): {abs(base_y[-1]):.4f}m")

print(f"\n⚡ Optimization Performance:")
print(f"  Success rate: {np.mean(data_log['success_flags'])*100:.1f}%")
print(f"  Avg optimization time: {np.mean(data_log['optimization_times'])*1000:.2f}ms")
print(f"  Max optimization time: {np.max(data_log['optimization_times'])*1000:.2f}ms")
print(f"  Min optimization time: {np.min(data_log['optimization_times'])*1000:.2f}ms")
print(f"  Real-time capable: {'✓ YES' if np.max(data_log['optimization_times']) < CONTROL_DT else '✗ NO'}")

print(f"\n🔧 Control Effort:")
total_torque = sum(np.mean(np.abs(data_log['applied_torques'][j])) for j in range(num_joints))
max_torque = max(np.max(np.abs(data_log['applied_torques'][j])) for j in range(num_joints))
print(f"  Total avg torque: {total_torque:.2f} N·m")
print(f"  Max torque (any joint): {max_torque:.2f} N·m")

print("\n" + "=" * 70)